In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



# BB84 without an attacker

This notebook simulates the BB84 quantum key distribution protocol in one sequential program. The comments mark the parts that belong to Alice and Bob.

Important assignment rule: the program does not use Python's standard random number generator. Every random bit is created by preparing the state `1/sqrt(2)(|0> + |1>)`, measuring it, and using the measurement outcome.

In [ ]:
backend = BasicSimulator()


def run_circuit_once(circuit):
    """Run a circuit once and return the measured bit string."""
    compiled = transpile(circuit, backend)
    result = backend.run(compiled, shots=1).result()
    return next(iter(result.get_counts()))


def quantum_random_bits(number_of_bits):
    """Generate random bits by measuring Hadamard-prepared qubits."""
    bits = []
    batch_size = 16

    while len(bits) < number_of_bits:
        current_batch_size = min(batch_size, number_of_bits - len(bits))
        circuit = QuantumCircuit(current_batch_size, current_batch_size)

        # Each qubit starts in |0>. Applying H gives 1/sqrt(2)(|0> + |1>).
        for qubit in range(current_batch_size):
            circuit.h(qubit)
            circuit.measure(qubit, qubit)

        bit_string = run_circuit_once(circuit)
        bits.extend(int(bit) for bit in bit_string)

    return bits


def prepare_alice_qubit(bit, basis):
    """Alice prepares one BB84 qubit.

    basis 0 = computational/Z basis: |0>, |1>
    basis 1 = diagonal/X basis: |+>, |->
    """
    circuit = QuantumCircuit(1, 1)

    if bit == 1:
        circuit.x(0)
    if basis == 1:
        circuit.h(0)

    return circuit


def bob_measures(circuit, basis):
    """Bob measures one qubit in his chosen basis."""
    if basis == 1:
        circuit.h(0)

    circuit.measure(0, 0)
    return int(run_circuit_once(circuit))


## Sequential BB84 program

Alice creates random data bits and random bases, prepares qubits, and sends them to Bob. Bob independently creates random measurement bases and measures the qubits. Afterwards Alice and Bob publicly compare only their bases, keep positions where the bases match, and reveal a small sample to check whether the quantum channel appears disturbed.

In [ ]:
# -----------------------------
# Protocol settings
# -----------------------------
NUMBER_OF_QUBITS = 80
TEST_SAMPLE_SIZE = 20
DISRUPTION_THRESHOLD = 0.15


# -----------------------------
# Alice: choose bits and bases
# -----------------------------
alice_bits = quantum_random_bits(NUMBER_OF_QUBITS)
alice_bases = quantum_random_bits(NUMBER_OF_QUBITS)


# -----------------------------
# Bob: choose measurement bases
# -----------------------------
bob_bases = quantum_random_bits(NUMBER_OF_QUBITS)
bob_results = []


# -----------------------------
# Quantum channel: Alice sends qubits directly to Bob
# -----------------------------
for index in range(NUMBER_OF_QUBITS):
    qubit = prepare_alice_qubit(alice_bits[index], alice_bases[index])
    bob_results.append(bob_measures(qubit, bob_bases[index]))


# -----------------------------
# Alice and Bob: public basis comparison and sifting
# -----------------------------
matching_basis_positions = [
    index
    for index in range(NUMBER_OF_QUBITS)
    if alice_bases[index] == bob_bases[index]
]

test_positions = matching_basis_positions[:TEST_SAMPLE_SIZE]
key_positions = matching_basis_positions[TEST_SAMPLE_SIZE:]

test_errors = sum(
    1
    for index in test_positions
    if alice_bits[index] != bob_results[index]
)
disruption_rate = test_errors / len(test_positions) if test_positions else 0
attacker_detected = disruption_rate > DISRUPTION_THRESHOLD

alice_key = [alice_bits[index] for index in key_positions]
bob_key = [bob_results[index] for index in key_positions]


# -----------------------------
# Results
# -----------------------------
print("BB84 without attacker")
print("Number of transmitted qubits:", NUMBER_OF_QUBITS)
print("Matching basis positions:", len(matching_basis_positions))
print("Test sample size:", len(test_positions))
print("Test errors:", test_errors)
print("Disruption rate:", disruption_rate)
print("Disruption threshold:", DISRUPTION_THRESHOLD)
print("Attacker detected:", attacker_detected)
print("Alice final key:", alice_key)
print("Bob final key:  ", bob_key)
print("Keys match:", alice_key == bob_key)
